# Stage 3 — DPO Alignment (Preference Optimization)
### Pull Stage 2 from the Hub → align to preferred answers → push the final model

**Notebook 3 of 4.**

### What this stage does, in plain words

After Stage 2 the model *can* answer. But "can answer" ≠ "answers the way we
want". Two answers can both be factually fine while one is crisp and professional
and the other is vague and generic.

Stage 2 couldn't fix that, because it only ever saw **one** gold answer per
question — it had no way to learn *preference*.

Here we show the model **pairs**: for the same question, a **chosen** (good)
answer and a **rejected** (worse) one. It learns to make chosen-style answers more
likely and rejected-style ones less likely.

| | Stage 2 (SFT) | Stage 3 (DPO) |
|---|---|---|
| Learns from | one gold answer | a **comparison** between two |
| Teaches | *what* to say | *which way* of saying it is better |
| Signal | "copy this" | "prefer this over that" |

### Concept: what DPO actually is

Classic **RLHF** needs two extra machines: train a separate *reward model* to
score answers, then run reinforcement learning against it. Complex and fragile.

**DPO (Direct Preference Optimization)** skips both. It proves you can optimize
the preference directly with a plain loss function — no reward model, no RL loop.
Just chosen/rejected pairs and gradient descent.

It keeps a frozen **reference model** (the Stage-2 model, with the adapter
switched off) as an anchor, so the model improves *without drifting away* from
everything it already learned. `beta` controls how tight that leash is.

## 1. Install libraries

In [1]:
# ============================================================
# Step 1. Install libraries
# ============================================================
!pip install -q unsloth
!pip install -q transformers trl datasets peft bitsandbytes accelerate sentencepiece protobuf huggingface_hub

## 2. Imports and GPU check

In [2]:
# ============================================================
# Step 2. Imports + GPU check
# ============================================================
import torch, json, gc
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig      # <-- DPO, not SFT
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda
GPU: Tesla T4


## Hugging Face login

Every stage pushes its merged model to the Hub, and the next stage pulls it back
down. So we log in once, here, at the top.

**Never paste a raw token into a notebook cell.** A token in a saved `.ipynb` is
a leaked credential — anyone with the file can push or delete on your account.
Use a **Colab secret** instead:

> Click the **key icon (🔑)** in the Colab left sidebar → **Add new secret** →
> Name: `HF_TOKEN`, Value: your token from
> https://huggingface.co/settings/tokens (needs **write** access) → toggle
> **Notebook access** on.

The cell below reads that secret automatically, and falls back to an interactive
prompt if it isn't set.

In [3]:
# ============================================================
# Hugging Face login
# ============================================================
from huggingface_hub import login, whoami

try:
    # Colab secret named HF_TOKEN (recommended).
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    # Fallback: paste the token when prompted.
    login()

HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Logged in as: Mohan143


## 3. Configuration

Two things to notice:

- `learning_rate = 5e-5` — **lowest of all three stages**. DPO is a delicate
  nudge, not a rebuild. Too high and the model degrades badly.
- `dpo_beta = 0.1` — the anchor strength. Explained in the concept block below.

In [4]:
# ============================================================
# Step 3. Configuration
# ============================================================
STAGE2_REPO = f"{HF_USERNAME}/hr-policy-assistant-stage2"   # <- input
FINAL_REPO  = f"{HF_USERNAME}/hr-policy-assistant-final"    # <- output
PRIVATE     = True

model_name     = STAGE2_REPO
max_seq_length = 512
dtype          = None
load_in_4bit   = True

# --- LoRA (fresh adapter for the preference stage) ---
lora_rank      = 16
lora_alpha     = 32
lora_dropout   = 0
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# --- DPO-specific ---
dpo_beta            = 0.1    # how tightly to stay near the reference model
max_prompt_length   = 256    # cap on the prompt part

# --- Training ---
learning_rate               = 5e-5   # LOWEST of the 3 stages - gentle nudge
num_train_epochs            = 3
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
warmup_steps                = 30
logging_steps               = 20
save_steps                  = 100
seed                        = 42

print(f"Loading base from : {STAGE2_REPO}")
print(f"Will push to      : {FINAL_REPO}")
print(f"DPO beta          : {dpo_beta}")

Loading base from : Mohan143/hr-policy-assistant-stage2
Will push to      : Mohan143/hr-policy-assistant-final
DPO beta          : 0.1


### Concept: `beta` — the most important DPO knob

`beta` controls how far the model is allowed to drift from the frozen Stage-2
reference model while chasing the preference signal.

| beta | Behaviour | Risk |
|---|---|---|
| **Low (0.01–0.05)** | Loose leash — big changes allowed | Model can drift and degrade; forgets Stage 2 |
| **0.1 (our value)** | Balanced — the standard default | Sensible starting point |
| **High (0.5+)** | Tight leash — stays very close to Stage 2 | Barely changes; DPO does almost nothing |

Think of it as: *"improve the answers — but don't wander off and become a
different model."*

If after training the answers look broken or nonsensical, `beta` was probably too
low (or the learning rate too high). If nothing changed at all, `beta` was too
high.

### Concept: quantization (what `load_in_4bit` actually does)

A model's weights are numbers. **Precision** is how many bits we use per number.

| Precision | Bits/weight | Memory for a 1.5B model | Note |
|---|---|---|---|
| float32 (full) | 32 | ~6.0 GB | Original training precision |
| float16 / bfloat16 (half) | 16 | ~3.0 GB | Standard for inference |
| **4-bit (NF4)** | **4** | **~0.9 GB** | What we use — fits a free T4 easily |

**Quantization** = storing those weights in fewer bits. It's like saving a photo
as a smaller JPEG: slightly less detail, dramatically less space.

- **NF4** ("4-bit NormalFloat") is a 4-bit format designed for the bell-curve
  shape that neural-network weights actually follow, so it loses less accuracy
  than naive 4-bit rounding.
- The quantized weights stay **frozen**. Maths still happens in 16-bit, so
  quality loss is small.
- **QLoRA** = *quantized base model* + *LoRA adapter trained on top*. That
  combination is what makes fine-tuning a 1.5B model possible on a free GPU.

Trade-off: 4-bit saves memory but is slightly slower per step and marginally
less precise. For fine-tuning on a T4, it's the right call.

## 4. Load the preference dataset

DPO data has exactly three fields:

| Field | Meaning |
|---|---|
| `prompt` | the question (wrapped in our chat template) |
| `chosen` | the answer we **want** — specific, professional, policy-grounded |
| `rejected` | a plausible but **worse** answer — vague, generic, unhelpful |

The `rejected` answer must be *realistic*, not nonsense. The model learns from the
**gap** between the two, so a strawman teaches it nothing useful.

Note the prompt uses the **exact same template as Stage 2** — that consistency is
what lets the preference signal land on the behaviour we actually trained.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [10]:
# ============================================================
# Step 4. Load DPO preference dataset (prompt / chosen / rejected)
# ============================================================
DATA_PATH = "/content/drive/MyDrive/AgenticAI/HR-Finetuning/preference_dataset.jsonl"

preference_data = []
try:
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            preference_data.append(json.loads(line))
    print(f"Loaded {len(preference_data)} preference pairs.")
except FileNotFoundError:
    print("File not found -> using a small embedded sample (replace with your real data).")
    preference_data = [
        {"prompt": "How many casual leaves can I take per year?",
         "chosen": "All full-time employees are entitled to 12 days of casual leave per calendar year. Please note that casual leave cannot be carried forward and will lapse on December 31st. You can apply through the HR portal.",
         "rejected": "You get some casual leaves every year. Check with HR for the exact number."},
        {"prompt": "What is the work from home policy?",
         "chosen": "The hybrid work policy allows eligible employees to work remotely for up to 2 days per week. Wednesday is a mandatory in-office day to support team collaboration. Requests should be submitted at least 24 hours in advance through the HR portal.",
         "rejected": "You can work from home sometimes. It depends on your manager."},
        {"prompt": "What is the notice period for resignation?",
         "chosen": "The notice period is 60 days for roles up to senior manager level and 90 days for director level and above. Notice must be served in full unless a buyout is approved by your department head and HR.",
         "rejected": "The notice period is usually a couple of months. Ask your manager about it."},
    ]

print(f"\nSample prompt  : {preference_data[0]['prompt']}")
print(f"Sample chosen  : {preference_data[0]['chosen'][:120]}...")
print(f"Sample rejected: {preference_data[0]['rejected'][:120]}...")

Loaded 302 preference pairs.

Sample prompt  : How many casual leaves can I take in a year?
Sample chosen  : Full-time employees are entitled to 12 days of casual leave per calendar year. Casual leave is meant for short, unplanne...
Sample rejected: You get some casual leaves. Check with HR about the exact number....


In [12]:
# ============================================================
# Step 4b. Wrap prompts in the SAME chat template used in Stage 2
# ============================================================
SYSTEM_PROMPT = "You are a helpful HR Policy Assistant."


def format_dpo(example):
    prompt = (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
              f"<|im_start|>user\n{example['prompt']}<|im_end|>\n"
              f"<|im_start|>assistant\n")
    return {"prompt": prompt,
            "chosen": example["chosen"],
            "rejected": example["rejected"]}


dataset = Dataset.from_list([format_dpo(ex) for ex in preference_data])
print(f"Formatted dataset: {len(dataset)} pairs\n")
print("PROMPT:\n" + dataset[0]["prompt"])
print("CHOSEN  :", dataset[0]["chosen"][:130])
print("REJECTED:", dataset[0]["rejected"][:130])

Formatted dataset: 302 pairs

PROMPT:
<|im_start|>system
You are a helpful HR Policy Assistant.<|im_end|>
<|im_start|>user
How many casual leaves can I take in a year?<|im_end|>
<|im_start|>assistant

CHOSEN  : Full-time employees are entitled to 12 days of casual leave per calendar year. Casual leave is meant for short, unplanned absences
REJECTED: You get some casual leaves. Check with HR about the exact number.


## 5. Load the Stage-2 model and attach a fresh LoRA adapter

**Where's the reference model?** We pass `ref_model=None`. With a LoRA setup, DPO
uses a clever trick: the *reference* is just this same base model with the adapter
**temporarily disabled**. One model in memory, two roles — which is exactly why
this fits on a free T4.

In [13]:
# ============================================================
# Step 5. Load Stage-2 model (4-bit) + NEW LoRA adapter
# ============================================================
print(f"Loading {model_name} for DPO training...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_name,
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = lora_rank,
    target_modules             = target_modules,
    lora_alpha                 = lora_alpha,
    lora_dropout               = lora_dropout,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = seed,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

Loading Mohan143/hr-policy-assistant-stage2 for DPO training...
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable params: 18,464,768


## 6. Build the DPOTrainer

> **trl version note.** This uses the modern API: hyperparameters live in
> `DPOConfig`, and the tokenizer is passed as `processing_class`. On older trl
> (< 0.12) use `tokenizer=tokenizer` instead, and move `beta` / `max_length` /
> `max_prompt_length` into the `DPOTrainer(...)` call.

In [14]:
# ============================================================
# Step 6. DPOTrainer
# ============================================================
trainer = DPOTrainer(
    model            = model,
    ref_model        = None,          # LoRA: reference = same base, adapter disabled
    processing_class = tokenizer,     # older trl: tokenizer=tokenizer
    train_dataset    = dataset,
    args = DPOConfig(
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_steps        = warmup_steps,
        num_train_epochs    = num_train_epochs,
        learning_rate       = learning_rate,
        beta                = dpo_beta,           # the anchor strength
        max_length          = max_seq_length,
        max_prompt_length   = max_prompt_length,
        fp16                = not torch.cuda.is_bf16_supported(),
        bf16                = torch.cuda.is_bf16_supported(),
        logging_steps       = logging_steps,
        save_steps          = save_steps,
        optim               = "adamw_8bit",
        weight_decay        = 0.01,
        lr_scheduler_type   = "linear",
        seed                = seed,
        output_dir          = "outputs/stage3_dpo",
        report_to           = "none",
    ),
)

print("DPOTrainer ready | beta =", dpo_beta, "| pairs:", len(dataset))

Extracting prompt in train dataset (num_proc=1):   0%|          | 0/302 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/302 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/302 [00:00<?, ? examples/s]

DPOTrainer ready | beta = 0.1 | pairs: 302


## 7. Train

**DPO logs different metrics than SFT — read these, not just the loss:**

| Metric | Meaning | Want |
|---|---|---|
| `rewards/chosen` | how much more likely the chosen answer became | rising |
| `rewards/rejected` | how much more likely the rejected answer became | **falling** |
| `rewards/margins` | chosen minus rejected — **the key number** | **rising, positive** |
| `rewards/accuracies` | % of pairs it correctly prefers | rising toward 1.0 |

If `margins` stays near zero, DPO isn't learning the preference.

In [15]:
# ============================================================
# Step 7. Train  (10-15 min on a T4)
# ============================================================
print("Starting Stage 3 DPO training...")
stats = trainer.train()

print("\nTraining complete.")
print(f"Runtime: {stats.metrics['train_runtime']:.0f} s")

# Pull the DPO-specific metrics from the last logged step.
last = [h for h in trainer.state.log_history if "rewards/margins" in h]
if last:
    m = last[-1]
    print(f"\nrewards/chosen    : {m.get('rewards/chosen'):.4f}   (want: rising)")
    print(f"rewards/rejected  : {m.get('rewards/rejected'):.4f}   (want: falling)")
    print(f"rewards/margins   : {m.get('rewards/margins'):.4f}   (want: positive & rising)")
    print(f"rewards/accuracies: {m.get('rewards/accuracies'):.4f}   (want: -> 1.0)")
else:
    print("No reward metrics logged - try more steps or lower logging_steps.")

Starting Stage 3 DPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 302 | Num Epochs = 3 | Total steps = 114
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
20,0.411023,0.843375,-0.136556,0.868750,0.979931,-326.121521,-53.083923,-0.128812,-0.447226
40,0.034496,3.066812,-1.682398,1.000000,4.749210,-296.124542,-68.732765,-0.038520,-0.373609
60,0.000972,4.267963,-4.204798,1.000000,8.472761,-292.206848,-94.311661,-0.176561,-0.695769
80,0.000490,4.335149,-4.952080,1.000000,9.287229,-308.929077,-101.594688,-0.120406,-0.753957
100,0.000339,4.244316,-5.129592,1.000000,9.373907,-286.306152,-102.925919,-0.192319,-0.789996


Unsloth: Restored added_tokens_decoder metadata in outputs/stage3_dpo/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/stage3_dpo/checkpoint-114/tokenizer_config.json.



Training complete.
Runtime: 628 s

rewards/chosen    : 4.2443   (want: rising)
rewards/rejected  : -5.1296   (want: falling)
rewards/margins   : 9.3739   (want: positive & rising)
rewards/accuracies: 1.0000   (want: -> 1.0)


## 8. Save, merge, and push the FINAL model

In [16]:
# ============================================================
# Step 8. Save adapter, merge, push the final model
# ============================================================
model.save_pretrained("stage3_lora_adapter")
tokenizer.save_pretrained("stage3_lora_adapter")
print("Adapter saved -> stage3_lora_adapter/")

print("\nMerging...")
model.save_pretrained_merged("final_hr_assistant_model", tokenizer, save_method="merged_16bit")
print("Merged -> final_hr_assistant_model/")

print(f"\nPushing to {FINAL_REPO} ...")
model.push_to_hub_merged(FINAL_REPO, tokenizer, save_method="merged_16bit", private=PRIVATE)
print(f"Done -> https://huggingface.co/{FINAL_REPO}")

Unsloth: Restored added_tokens_decoder metadata in stage3_lora_adapter/tokenizer_config.json.


Adapter saved -> stage3_lora_adapter/

Merging...


Unsloth: Restored added_tokens_decoder metadata in final_hr_assistant_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `final_hr_assistant_model`: 100%|██████████| 1/1 [03:28<00:00, 208.35s/it]


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t-final/model.safetensors:   0%|          | 7.88MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [05:23<00:00, 323.11s/it]


Unsloth: Merge process complete. Saved to `/content/Mohan143/hr-policy-assistant-final`
Done -> https://huggingface.co/Mohan143/hr-policy-assistant-final


## 9. Quick check

A full three-way comparison lives in Notebook 4. This is just a smoke test.

In [17]:
# ============================================================
# Step 9. Smoke test
# ============================================================
FastLanguageModel.for_inference(model)

for q in ["How many casual leaves can I take per year?",
          "What is the notice period for resignation?"]:
    prompt = (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
              f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
    print("="*80)
    print(f"Q: {q}\nA: {resp[:320]}")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Q: How many casual leaves can I take per year?
A: The annual casual leave allowance is 5 days per year, which may be claimed on a non-consecutive basis as long as total casual leave does not exceed 15 working days in a calendar year. The casual leave window runs from April 1st to March 31st, during which you can submit requests for one or more days. Casual leave reque
Q: What is the notice period for resignation?
A: The standard notice period for resignation is 30 days for permanent employees with 1+ year tenure, 25 days for employees with 0.5+ to 1 year tenure, and 20 days for employees with 0 to 0.5 year tenure. The notice period begins from the date nominated by the employee and must be completed before the last working day of 


## Understanding the DPO loss function (this one is different!)

Stages 1 and 2 used cross-entropy: *"predict the next token."* **DPO's loss is a
completely different shape** — it doesn't grade single tokens, it grades a
**comparison**.

### The formula

```text
loss = -log σ( β · [ (log π(chosen)  - log π_ref(chosen))
                   - (log π(rejected) - log π_ref(rejected)) ] )
```

Scary-looking, so let's take it apart.

### Piece by piece

**1. `log π(answer) - log π_ref(answer)` — the "reward".**
`π` is our model being trained; `π_ref` is the frozen Stage-2 reference. This asks:
*"has our model made this answer **more** likely than the reference did?"*
Positive = yes, more likely. This difference is what trl logs as `rewards/chosen`
and `rewards/rejected`.

**2. `reward(chosen) - reward(rejected)` — the margin.**
This is the heart of it: *"did we push the good answer up **more** than the bad
one?"* Big positive number = we're doing exactly what we want. This is
`rewards/margins`.

**3. `β ·` — the anchor.**
Scales how much that margin is allowed to matter versus staying near the
reference. Low beta = drift freely. High beta = barely move.

**4. `σ(...)` — the sigmoid.**
Squashes any number into the range 0–1, turning the margin into a probability:
*"how confident are we that chosen beats rejected?"*

**5. `-log(...)` — same trick as cross-entropy.**
Confident and correct → loss near 0. Confident and wrong → loss explodes.

### The one-sentence version

> **DPO loss is low when the model makes the chosen answer more likely than the
> rejected one — relative to where it started.**

### Reading your numbers

| Signal | Meaning |
|---|---|
| Loss falling from ~0.69 toward 0 | Working. (0.69 = `-log(0.5)` = pure coin-flip, the starting point) |
| `rewards/margins` rising, positive | **The main success signal** |
| `rewards/accuracies` → 1.0 | It reliably prefers chosen over rejected |
| `rewards/rejected` going negative | Good — bad answers are being suppressed |
| Margins stuck near 0 | Not learning: raise LR slightly, or your pairs are too similar |
| Answers became gibberish | Drifted too far: raise `beta` or lower the LR |

### The catch worth knowing

DPO optimizes **"preferred-ness"**, not correctness. A model can learn to sound
more polished while quietly becoming *less* accurate — the loss won't tell you.
That's precisely why **Notebook 4** compares all three models on real questions.

---

## Pipeline complete

| Stage | Learned | Loss type |
|---|---|---|
| 1 | the language of HR | cross-entropy (self-supervised) |
| 2 | how to answer | cross-entropy (supervised) |
| 3 | which answers are better | DPO preference loss |

**Next:** open **`Stage4_Model_Comparison.ipynb`** to judge whether it all
actually worked.